Importation des bibliothèque

In [1]:
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt 
from biopandas.pdb import PandasPdb
import nglview as nv
from scipy import stats

In [ ]:
def view_structure(file_pdb):
    # 1. Charger le fichier généré
    view = nv.show_file(file_pdb)
    # 2. Vider les représentations par défaut
    view.clear_representations()
    # 3. Afficher toute la molécule de manière fine
    view.add_representation('licorice', selection='all', color='element')
    # 4. Mettre en surbrillance spéciale les atomes de Phosphore (les "noeuds" des liaisons phosphodiester)
    view.add_representation('spacefill', selection='_P', color='orange', radius=0.8)
    # 5. Rajouter un "tube" qui suit et met en évidence de façon continue le squelette phosphodiester (backbone)
    view.add_representation('tube', selection='backbone', color='red', radius=0.2)
    
    # # 6. Ajout d'un dégradé de couleur pour voir l'orientation 5' -> 3'
    # # Le schéma 'resindex' colore du bleu (début/5') au rouge (fin/3')
    view.add_representation('cartoon', selection='nucleic', color='resindex')
    
    # 7. Centrer la vue
    view.center()
    # Afficher l'interface
    return view

In [ ]:
view_structure("resultat/optimized_rasp_1774627195.pdb")

In [ ]:
ppdb_df =  PandasPdb().read_pdb("fichier_arn/initial_optimized.pdb")
atom_df = ppdb_df.df["ATOM"]
atom_df.head(20)

In [ ]:
from Bio.PDB import Structure, Model, Chain, Residue, Atom, PDBIO

def generer_structure_depart_c3(seq, distance_moy=5.8, output="initial_c3.pdb"):
    # Création de la hiérarchie
    structure = Structure.Structure("RNA")
    model = Model.Model(0)
    chain = Chain.Chain("A")
    
    for i, nt in enumerate(seq):
        # Création du résidu (nucléotide)
        res = Residue.Residue((" ", i+1, " "), nt, " ")
        # Création de l'atome C3' à sa position x
        atom = Atom.Atom("C3'", [i * distance_moy, 0, 0], 0, 0, " ", "C3'", i+1, "C")
        res.add(atom)
        chain.add(res)
        
    model.add(chain)
    structure.add(model)
    
    # Sauvegarde
    io = PDBIO()
    io.set_structure(structure)
    io.save(output)


def voir_configuration_c3(fichier_pdb):
    """
    Visualisation NGL adaptée au modèle C3'-seulement.
    """
    view = nv.show_file(fichier_pdb)
    view.clear_representations()
    view.add_representation("ball+stick", selection="all", color="cyan", radius=0.25)
    view.center()
    return view


# Exemple rapide :
generer_structure_depart_c3(seq="ACGU"*30, distance_moy=5.8)
voir_configuration_c3("fichier_arn/initial_c3_beads.pdb")

In [ ]:
def test_shapiro(df_path):
    df = pd.read_csv(df_path)
    stat, p = stats.shapiro(df['Distance'])
    print(f"Statistique de Shapiro-Wilk : {stat:.3f}")
    print(f"p-value : {p}")
    alpha = 0.05
    if p > alpha:
        print("On ne peut pas rejeter l'hypothèse nulle (H0). Les données semblent suivre une loi normale.")
    else:
        print("On rejette l'hypothèse nulle (H0). Les données ne suivent probablement pas une loi normale.")

test_shapiro("distances_1-5.csv")

In [ ]:
import matplotlib.pyplot as plt
import scipy.stats as stats

# Création du graphique Q-Q
df = pd.read_csv("distances_1-5.csv")
data = df["Distance"]
stats.probplot(data, dist="norm", plot=plt)
plt.title("Graphique Q-Q pour vérifier la normalité")
plt.show()

In [1]:
from classe.BeadSpringsRASPOptimizer import BeadSpringsRASPOptimizer

model = BeadSpringsRASPOptimizer("fichier_arn/initial_c3_beads.pdb")
model.run_optimization()


Utilisation du device : cuda
🚀 Début de l'optimisation (Adam, 5 cycles de 100 epochs)...

--- Cycle 1/5 ---
Epoch   0 | Total: -239.47 | RASP: -239.47 | FENE: -0.00
Epoch  99 | Total: -29926.17 | RASP: -165.03 | FENE: -29761.15
Secousse aléatoire ! (Bruit translation: 1.50Å, rotation: 0.50rad)

--- Cycle 2/5 ---
Epoch   0 | Total: -6628.17 | RASP: -169.84 | FENE: -6458.33
Epoch  99 | Total: -26553.67 | RASP: -171.03 | FENE: -26382.64
Secousse aléatoire ! (Bruit translation: 1.12Å, rotation: 0.38rad)

--- Cycle 3/5 ---
Epoch   0 | Total: -14280.30 | RASP: -157.71 | FENE: -14122.59
Epoch  99 | Total: -30060.35 | RASP: -161.67 | FENE: -29898.68
Secousse aléatoire ! (Bruit translation: 0.75Å, rotation: 0.25rad)

--- Cycle 4/5 ---
Epoch   0 | Total: -14766.62 | RASP: -159.76 | FENE: -14606.87
Epoch  99 | Total: -29839.55 | RASP: -166.64 | FENE: -29672.90
Secousse aléatoire ! (Bruit translation: 0.38Å, rotation: 0.12rad)

--- Cycle 5/5 ---
Epoch   0 | Total: -21413.10 | RASP: -164.80 | FENE:

/home/erwann/.conda/envs/Stage/lib/python3.10/site-packages/biopandas/pdb/pandas_pdb.py:732: UserWarning: Column rasp_type is not an expected column and will be skipped.
  warn(


In [2]:
def voir_configuration_c3(fichier_pdb):
    """
    Visualisation NGL adaptée au modèle C3'-seulement.
    """
    view = nv.show_file(fichier_pdb)
    view.clear_representations()
    view.add_representation("ball+stick", selection="all", color="cyan", radius=0.25)
    view.center()
    return view

In [7]:
voir_configuration_c3("output_bead_optimized.pdb")

NGLWidget()